# VisionGuard AI — Home Defect AI Dataset Preparation Pipeline (Hardened v2.0)

**Project:** VisionGuard AI — Home & Infrastructure Defect Inspection Platform  
**Target Component:** Separate Home Defect Detection Model (`ai/models/home_best.pt`)  
**Target Canonical Classes:**
- `0: wall_crack`
- `1: damp_or_water_damage`
- `2: peeling_paint_or_plaster`
- `3: spalling_or_surface_damage`

---

### Non-Negotiable Engineering Guardrails
1. **Model Decoupling:** The working road-defect YOLO model (`best.pt`, trained on RDD2022) is completely untouched.
2. **Transparent Annotation Lineage:** Mask-derived bounding boxes (e.g. from Zenodo binary masks) are strictly tracked as `mask_derived_bbox` in `dataset_manifest.json`; they are never claimed to be manual bounding boxes.
3. **Ambiguity Elimination Policy:** Generic dirt stains and uneven plaster roughness are NOT automatically treated as dampness or spalling. Ambiguous tags are pruned from ground truth.
4. **Leakage-Resistant Group Splitting:** Images sharing the same building, property, sequence, or augmented base-name are strictly grouped into the same split (train, val, or test).
5. **Strict Pre-Training Readiness Gate:** The pipeline asserts that train, val, and test splits are all non-empty, and that all 4 canonical classes possess verified ground-truth annotations. If any check fails, the dataset is marked **`NOT READY FOR TRAINING`** and packaging is blocked.


## Step 1: Runtime & Hardware Verification

Inspect the Colab or local environment to verify Python version, available memory, and storage space.


In [ ]:
# 1. System diagnostics and hardware capabilities
import os
import sys
import shutil
import platform

print("=== VisionGuard AI: System Diagnostics ===")
print(f"Platform:         {platform.platform()}")
print(f"Python Version:   {sys.version.split()[0]}")

working_dir = os.getcwd()
total_b, used_b, free_b = shutil.disk_usage(working_dir)
print(f"Working Directory: {working_dir}")
print(f"Free Disk Space:   {free_b / (1024**3):.2f} GB")

try:
    import torch
    cuda_avail = torch.cuda.is_available()
    print(f"PyTorch CUDA:     {cuda_avail}")
    if cuda_avail:
        print(f"GPU Device:       {torch.cuda.get_device_name(0)}")
        print(f"Dedicated VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
except ImportError:
    print("PyTorch not installed in the current environment (CPU execution mode).")


## Step 2: Dependency Verification & Installation

Verify and install essential computer vision, data manipulation, and image hashing packages.


In [ ]:
# 2. Dependency verification
import sys
import subprocess

required_pkgs = ["opencv-python", "pillow", "matplotlib", "pyyaml", "imagehash", "requests", "tqdm"]
missing = []
for pkg in required_pkgs:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing dependencies: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import cv2
import PIL
import yaml
import imagehash
import requests
import matplotlib

print("=== Environment Baseline Verified ===")
print(f"OpenCV Version:    {cv2.__version__}")
print(f"Pillow Version:    {PIL.__version__}")
print(f"ImageHash Version: {imagehash.__version__}")
print(f"PyYAML Version:    {yaml.__version__}")
print(f"Matplotlib Version:{matplotlib.__version__}")
print("[PASSED] All dataset preparation dependencies verified.")


## Step 3: Dataset Provenance, Licensing & Grouping Audit

Audit candidate sources for exact origin, license restrictions, environment representation, and grouping metadata.

### Audited Dataset Sources
1. **Zenodo Record 18458458 (Residential Masonry Cracks)**:
   - *DOI*: `10.5281/zenodo.18458458` | *Authors*: Storey & Snowball (Northumbria Univ / ICE / IStructE)
   - *License*: **CC BY 4.0** | *Environment*: Low-rise residential masonry buildings in the UK.
   - *Format*: 100 RGB JPEG images (3000x4000) + 100 binary PNG crack masks.
   - *Target Defect*: `wall_crack` exclusively.
   - *CRITICAL GROUPING LIMITATION*: UK GDPR compliance required manual pixelation of street names and house numbers, and complete stripping of EXIF metadata. Filenames follow a flat sequence (`masonry_crack_001` to `100`). **No site or property identifiers are provided.** Grouping must treat each image as its own distinct capture (`group_id = stem`). Property-level grouping cannot be established for this source alone without the authors' unredacted property logs.
2. **DamSegment (Mendeley Data z5z6gtt5t4/1)**:
   - *DOI*: `10.17632/z5z6gtt5t4.1` | *Authors*: Gharehbaghi et al. (University of Kansas)
   - *License*: **CC BY 4.0** | *Environment*: Concrete civil structures & retaining walls.
   - *Format*: 1,500 detection images with YOLO `.txt` annotations (`crack`, `spalling`).
   - *Target Defects*: `wall_crack` and `spalling_or_surface_damage`.
   - *Ingestion Constraint*: Direct automated script downloads are blocked by Elsevier S3 authorization cookies. Requires user-supplied archive (`damsegment.zip`).
3. **Roboflow Universe: Surface Defects in Heritage**:
   - *Workspace*: `defects-dataset` | *Project*: `surface-defects-in-heritage`
   - *License*: **CC BY 4.0** | *Environment*: Building facades, masonry, plastered walls.
   - *Format*: 3,310 annotated images in YOLO format.
   - *Target Defects*: `wall_crack`, `damp_or_water_damage`, `peeling_paint_or_plaster`, `spalling_or_surface_damage`.
4. **Roboflow Universe: Internal Wall Defect**:
   - *Workspace*: `chew-poh-yee` | *Project*: `internal-wall-defect`
   - *License*: **CC BY 4.0** | *Environment*: Domestic living space interiors.
   - *Format*: 3,279 images in YOLO format (`peeling paint`, `stain marks`, `rough surface`).
   - *Target Defects*: `peeling_paint_or_plaster` (`peeling paint`). Non-structural and ambiguous tags are pruned.


In [ ]:
# 3. Provenance and Licensing Registry
DATASET_PROVENANCE = {
    "zenodo_residential_cracks": {
        "title": "Annotated Image Dataset of Masonry Building Cracks",
        "url": "https://zenodo.org/records/18458458",
        "doi": "10.5281/zenodo.18458458",
        "license": "CC BY 4.0",
        "commercial_use_permitted": True,
        "environment": "Residential low-rise masonry buildings (UK)",
        "annotation_type": "binary_segmentation_masks",
        "supported_classes": ["wall_crack"],
        "grouping_metadata_available": False,
        "grouping_note": "EXIF stripped and PII pixelated for UK GDPR. Flat numbering (001-100). No property metadata.",
        "download_urls": {
            "images": "https://zenodo.org/records/18458458/files/masonry_crack_images.zip?download=1",
            "masks": "https://zenodo.org/records/18458458/files/masonry_crack_masks.zip?download=1"
        }
    },
    "mendeley_damsegment": {
        "title": "DamSegment: Structural Concrete Crack & Spalling Dataset",
        "url": "https://data.mendeley.com/datasets/z5z6gtt5t4/1",
        "doi": "10.17632/z5z6gtt5t4.1",
        "license": "CC BY 4.0",
        "commercial_use_permitted": True,
        "environment": "Concrete structures & infrastructure",
        "annotation_type": "yolo_bounding_boxes_and_masks",
        "supported_classes": ["wall_crack", "spalling_or_surface_damage"],
        "grouping_metadata_available": True,
        "grouping_note": "Sequence identifiers in filename stem."
    },
    "roboflow_heritage_defects": {
        "title": "Surface Defects in Heritage Dataset",
        "url": "https://universe.roboflow.com/defects-dataset/surface-defects-in-heritage",
        "license": "CC BY 4.0",
        "commercial_use_permitted": True,
        "environment": "Building facades, plastered walls, masonry",
        "annotation_type": "yolo_bounding_boxes",
        "supported_classes": ["wall_crack", "damp_or_water_damage", "peeling_paint_or_plaster", "spalling_or_surface_damage"],
        "grouping_metadata_available": True,
        "grouping_note": "Roboflow hash prefix groups augmented crops of the same base image."
    },
    "roboflow_internal_wall": {
        "title": "Internal Wall Defect Dataset",
        "url": "https://universe.roboflow.com/chew-poh-yee/internal-wall-defect",
        "license": "CC BY 4.0",
        "commercial_use_permitted": True,
        "environment": "Interior domestic walls & plaster",
        "annotation_type": "yolo_bounding_boxes",
        "supported_classes": ["peeling_paint_or_plaster"],
        "grouping_metadata_available": True,
        "grouping_note": "Roboflow hash prefix groups augmented crops of the same base image."
    }
}

print("=== Verified Source Registry ===")
for k, v in DATASET_PROVENANCE.items():
    print(f"[{k}]")
    print(f"  Title:    {v['title']}")
    print(f"  License:  {v['license']} (Commercial Allowed: {v['commercial_use_permitted']})")
    print(f"  Domain:   {v['environment']}")
    print(f"  Classes:  {v['supported_classes']}")
    print(f"  Grouping: {v['grouping_note']}")


## Step 4: Ephemeral Workspace Setup & Multi-Path Ingestion

Establishes workspace folders and handles multi-source ingestion:
- **Direct Automated Download**: Attempts direct HTTP download for accessible sources (Zenodo).
- **Google Drive / Local Storage Discovery**: Automatically searches `/content/`, `/content/drive/MyDrive/VisionGuard_Artifacts/home_raw_data/`, and `./data` for user-supplied ZIP archives.
- **Fault Tolerance**: If an external server (e.g., Zenodo) experiences transient 504 timeouts, the notebook logs the pending status and continues.


In [ ]:
# 4. Storage paths and multi-source ingestion
from pathlib import Path
import urllib.request
import zipfile
import time

BASE_DIR = Path("/content") if Path("/content").exists() else Path("./home_dataset_workspace")
RAW_DIR = BASE_DIR / "raw_downloads"
EXTRACT_DIR = BASE_DIR / "extracted_sources"
PROCESSED_DIR = BASE_DIR / "visionguard_home_dataset"

for d in [RAW_DIR, EXTRACT_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Raw storage:       {RAW_DIR}")
print(f"Extracted storage: {EXTRACT_DIR}")
print(f"Output dataset:    {PROCESSED_DIR}")

def download_with_retry(url: str, dest_path: Path, max_retries: int = 3, timeout_sec: int = 60):
    if dest_path.is_file() and dest_path.stat().st_size > 1000:
        print(f"[EXISTS] {dest_path.name} already downloaded ({dest_path.stat().st_size / (1024*1024):.2f} MB).")
        return True
    
    for attempt in range(1, max_retries + 1):
        print(f"[DOWNLOAD ATTEMPT {attempt}/{max_retries}] {dest_path.name}...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (VisionGuard AI Dataset Pipeline)'})
            with urllib.request.urlopen(req, timeout=timeout_sec) as response, open(dest_path, 'wb') as out_file:
                shutil.copyfileobj(response, out_file)
            print(f"[COMPLETED] {dest_path.name} ({dest_path.stat().st_size / (1024*1024):.2f} MB)")
            return True
        except Exception as e:
            print(f"  Warning: Attempt {attempt} failed ({e}).")
            if attempt < max_retries:
                time.sleep(3)
    print(f"[FAILED] Could not download {dest_path.name} automatically.")
    return False

# 1. Zenodo Ingestion Attempt
zenodo_img_zip = RAW_DIR / "zenodo_masonry_crack_images.zip"
zenodo_mask_zip = RAW_DIR / "zenodo_masonry_crack_masks.zip"

zenodo_img_ok = download_with_retry(DATASET_PROVENANCE["zenodo_residential_cracks"]["download_urls"]["images"], zenodo_img_zip)
zenodo_mask_ok = download_with_retry(DATASET_PROVENANCE["zenodo_residential_cracks"]["download_urls"]["masks"], zenodo_mask_zip)

# 2. Search for Mendeley DamSegment & Roboflow Archives
GDRIVE_RAW_DIR = Path("/content/drive/MyDrive/VisionGuard_Artifacts/home_raw_data")
search_locations = [RAW_DIR, BASE_DIR, GDRIVE_RAW_DIR, Path("./data")]

def find_archive(patterns):
    for loc in search_locations:
        if not loc.is_dir():
            continue
        for p in loc.iterdir():
            if any(pat.lower() in p.name.lower() for pat in patterns) and p.suffix.lower() == ".zip":
                return p
    return None

damsegment_zip = find_archive(["damsegment", "z5z6gtt5t4"])
heritage_zip = find_archive(["heritage", "surface-defects"])
internal_wall_zip = find_archive(["internal_wall", "internal-wall"])

print("\n=== Ingestion Status Summary ===")
print(f"Zenodo Masonry Images:    {'FOUND' if zenodo_img_zip.is_file() else 'PENDING / MANUAL DOWNLOAD REQUIRED'}")
print(f"Zenodo Masonry Masks:     {'FOUND' if zenodo_mask_zip.is_file() else 'PENDING / MANUAL DOWNLOAD REQUIRED'}")
print(f"Mendeley DamSegment:      {damsegment_zip if damsegment_zip else 'NOT FOUND (Optional: place in raw_downloads/ or Drive)'}")
print(f"Roboflow Heritage Defects:{heritage_zip if heritage_zip else 'NOT FOUND (Optional: place in raw_downloads/ or Drive)'}")
print(f"Roboflow Internal Wall:   {internal_wall_zip if internal_wall_zip else 'NOT FOUND (Optional: place in raw_downloads/ or Drive)'}")


## Step 5: Dynamic Archive Inspection & Safe Extraction Workflow

Inspect user-supplied ZIP archives before extraction:
- Detect actual internal class names from `data.yaml`, `classes.txt`, or directory names.
- Verify presence of license files (`LICENSE*`, `README*`).
- Detect annotation structure (YOLO text boxes, Pascal VOC XML, or binary masks).
- Avoid guessing class IDs or assuming compatibility without verification.


In [ ]:
# 5. Dynamic archive inspection and extraction
def inspect_and_extract_archive(zip_path: Path, target_dir: Path, source_key: str):
    if not zip_path or not zip_path.is_file():
        print(f"[SKIP] Archive not found for {source_key}: {zip_path}")
        return None
        
    print(f"\n--- Inspecting Archive: {zip_path.name} ({zip_path.stat().st_size / (1024*1024):.2f} MB) ---")
    inspection = {
        "source_key": source_key,
        "zip_path": str(zip_path),
        "classes_found": [],
        "has_license": False,
        "format": "unknown",
        "file_count": 0
    }
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        namelist = zf.namelist()
        inspection["file_count"] = len(namelist)
        
        # Check for License or Readme
        license_files = [n for n in namelist if any(k in n.lower() for k in ["license", "readme"])]
        if license_files:
            inspection["has_license"] = True
            print(f"  Found license/documentation file: {license_files[0]}")
            
        # Check for data.yaml or classes.txt
        yaml_files = [n for n in namelist if n.lower().endswith("data.yaml") or n.lower().endswith("data.yml")]
        classes_files = [n for n in namelist if n.lower().endswith("classes.txt")]
        
        if yaml_files:
            try:
                content = zf.read(yaml_files[0]).decode('utf-8', errors='ignore')
                cfg = yaml.safe_load(content)
                names = cfg.get("names", [])
                if isinstance(names, dict):
                    inspection["classes_found"] = [names[k] for k in sorted(names.keys())]
                elif isinstance(names, list):
                    inspection["classes_found"] = names
                inspection["format"] = "yolo_yaml"
                print(f"  Parsed {yaml_files[0]}: Detected classes = {inspection['classes_found']}")
            except Exception as e:
                print(f"  Warning reading YAML: {e}")
        elif classes_files:
            try:
                content = zf.read(classes_files[0]).decode('utf-8', errors='ignore')
                inspection["classes_found"] = [l.strip() for l in content.splitlines() if l.strip()]
                inspection["format"] = "classes_txt"
                print(f"  Parsed {classes_files[0]}: Detected classes = {inspection['classes_found']}")
            except Exception as e:
                print(f"  Warning reading classes.txt: {e}")
                
        # Safe extraction
        target_dir.mkdir(parents=True, exist_ok=True)
        zf.extractall(target_dir)
        print(f"  Extracted {len(namelist)} files -> {target_dir}")
        
    return inspection

# Extract Zenodo
zenodo_extract = EXTRACT_DIR / "zenodo_residential"
if zenodo_img_zip.is_file():
    zenodo_extract.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zenodo_img_zip, 'r') as zf:
        zf.extractall(zenodo_extract / "images")
if zenodo_mask_zip.is_file():
    with zipfile.ZipFile(zenodo_mask_zip, 'r') as zf:
        zf.extractall(zenodo_extract / "masks")

# Inspect and extract optional user-supplied archives
archive_inspections = {}
if damsegment_zip:
    archive_inspections["mendeley_damsegment"] = inspect_and_extract_archive(
        damsegment_zip, EXTRACT_DIR / "mendeley_damsegment", "mendeley_damsegment"
    )
if heritage_zip:
    archive_inspections["roboflow_heritage"] = inspect_and_extract_archive(
        heritage_zip, EXTRACT_DIR / "roboflow_heritage", "roboflow_heritage"
    )
if internal_wall_zip:
    archive_inspections["roboflow_internal_wall"] = inspect_and_extract_archive(
        internal_wall_zip, EXTRACT_DIR / "roboflow_internal_wall", "roboflow_internal_wall"
    )

print("\nArchive inspection complete. Ready for annotation parsing.")


## Step 6: Binary Segmentation Mask to Bounding Box Derivation (Zenodo)

Extract localized bounding boxes from Zenodo's binary masks using contour boundary analysis (`cv2.findContours`).  
**Lineage Guardrail:** Every derived bounding box is assigned `annotation_type: "mask_derived_bbox"`, ensuring transparent provenance.


In [ ]:
# 6. Mask to YOLO bounding box converter
import numpy as np

def mask_to_yolo_bboxes(mask_path: Path, min_area_px: int = 40):
    """
    Converts binary crack mask (white=crack, black=background) into
    normalized YOLO bounding boxes: [class_id, x_center, y_center, width, height].
    Canonical class_id for wall_crack is 0.
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    h, w = mask.shape
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bboxes = []
    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        if (bw * bh) < min_area_px:
            continue  # Suppress single-pixel sensor noise
            
        x_c = (x + bw / 2.0) / float(w)
        y_c = (y + bh / 2.0) / float(h)
        norm_w = bw / float(w)
        norm_h = bh / float(h)
        
        # Clamp bounds strictly to [0, 1]
        x_c = min(max(x_c, 0.0), 1.0)
        y_c = min(max(y_c, 0.0), 1.0)
        norm_w = min(max(norm_w, 0.0), 1.0)
        norm_h = min(max(norm_h, 0.0), 1.0)
        
        bboxes.append({
            "class_id": 0,  # 0: wall_crack
            "class_name": "wall_crack",
            "bbox": [x_c, y_c, norm_w, norm_h],
            "pixel_bbox": [x, y, bw, bh],
            "annotation_type": "mask_derived_bbox"
        })
    return bboxes

print("Mask-to-YOLO bounding box conversion function registered.")


## Step 7: Canonical Taxonomy Mapping & Ambiguity Auditing

Map source annotations to VisionGuard AI's 4 canonical classes:
- `0: wall_crack`
- `1: damp_or_water_damage`
- `2: peeling_paint_or_plaster`
- `3: spalling_or_surface_damage`

### Strict Ambiguity Elimination Policy:
- **Generic Stains**: A surface stain (coffee, grease, superficial dirt) is NOT dampness. Only verified moisture indications (`dampness`, `rising damp`, `water seepage`, `water_damage`, `efflorescence`) are mapped to class 1.
- **Surface Roughness**: "Rough and patchy surface" (uneven plaster texture) is NOT structural spalling. Only genuine spalling (`spalling`, `concrete spalling`, `brick spalling`) is mapped to class 3.
- **Tradesman Marks**: `paint drips`, `pin holes`, and `trowel marks` are discarded to eliminate false alarms.


In [ ]:
# 7. Canonical taxonomy and ambiguity filtering rules
CANONICAL_CLASSES = {
    0: "wall_crack",
    1: "damp_or_water_damage",
    2: "peeling_paint_or_plaster",
    3: "spalling_or_surface_damage",
}

STRICT_LABEL_MAPPINGS = {
    # Class 0: wall_crack
    "crack": 0,
    "cracks": 0,
    "major crack": 0,
    "minor crack": 0,
    "masonry_crack": 0,
    "concrete crack": 0,
    
    # Class 1: damp_or_water_damage (Strict moisture only)
    "dampness": 1,
    "damp": 1,
    "rising damp": 1,
    "water seepage": 1,
    "water_damage": 1,
    "water damage": 1,
    "efflorescence": 1,
    
    # Class 2: peeling_paint_or_plaster
    "peeling paint": 2,
    "peeling_paint": 2,
    "peeling": 2,
    "flaking plaster": 2,
    "flaking_plaster": 2,
    "blistering paint": 2,
    
    # Class 3: spalling_or_surface_damage (Substrate loss)
    "spalling": 3,
    "concrete spalling": 3,
    "brick spalling": 3,
    "masonry spalling": 3,
}

# Ambiguous classes flagged and excluded from positive defect training
AMBIGUOUS_OR_DISCARDED = [
    "stain", "stain marks", "rough and patchy surface", "rough surface",
    "paint drips", "pin holes", "trowels marks", "floor", "window", "door", "vegetation"
]

def map_raw_label(raw_name: str):
    norm = raw_name.strip().lower()
    if norm in STRICT_LABEL_MAPPINGS:
        cid = STRICT_LABEL_MAPPINGS[norm]
        return cid, CANONICAL_CLASSES[cid], "accepted"
    if norm in AMBIGUOUS_OR_DISCARDED:
        return None, norm, "discarded_ambiguous"
    return None, norm, "unrecognized"

print("Canonical Taxonomy:")
for cid, cname in CANONICAL_CLASSES.items():
    print(f"  Class {cid}: {cname}")
print(f"\nStrictly Mapped Labels:    {len(STRICT_LABEL_MAPPINGS)}")
print(f"Ambiguous / Pruned Labels: {len(AMBIGUOUS_OR_DISCARDED)}")


## Step 8: Ingestion Engine & Perceptual Deduplication

Ingest all available extracted datasets into a unified in-memory candidate list, detect exact and near duplicates (pHash Hamming distance $\le 2$), and remove duplicates.

### Grouping Strategy:
- **Zenodo**: No property/site metadata exists (GDPR redacted). Each image is an independent group (`group_id = f"zenodo_{stem}"`).
- **DamSegment**: Groups sequential camera bursts sharing the same numerical prefix.
- **Roboflow**: Groups augmented crops sharing the same base image stem before `.rf.`.


In [ ]:
# 8. Unified candidate parsing and deduplication
import hashlib
from PIL import Image

def compute_image_hashes(img_path: Path):
    try:
        with open(img_path, 'rb') as f:
            md5_hash = hashlib.md5(f.read()).hexdigest()
        with Image.open(img_path) as img:
            phash_val = imagehash.phash(img)
        return md5_hash, phash_val
    except Exception:
        return None, None

candidates = []
flagged_ambiguous_log = []

# 1. Ingest Zenodo
zenodo_img_dir = zenodo_extract / "images"
zenodo_mask_dir = zenodo_extract / "masks"

if zenodo_img_dir.is_dir() and zenodo_mask_dir.is_dir():
    mask_map = {p.stem.replace("_mask", ""): p for p in zenodo_mask_dir.glob("*.png")}
    for img_p in sorted(zenodo_img_dir.glob("*.*")):
        if img_p.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        stem = img_p.stem
        if stem in mask_map:
            boxes = mask_to_yolo_bboxes(mask_map[stem])
            if boxes:
                # Defensible grouping: Zenodo has no property IDs, so each image is its own capture group
                candidates.append({
                    "id": stem,
                    "source": "zenodo",
                    "group_id": f"zenodo_{stem}",
                    "img_path": img_p,
                    "annotations": boxes,
                    "annotation_type": "mask_derived_bbox",
                    "original_label": "crack_mask"
                })

# 2. Ingest DamSegment if extracted
damsegment_dir = EXTRACT_DIR / "mendeley_damsegment"
if damsegment_dir.is_dir():
    # Discover class map from inspection if available
    archive_classes = archive_inspections.get("mendeley_damsegment", {}).get("classes_found", ["crack", "spalling"])
    for lbl_p in damsegment_dir.rglob("*.txt"):
        if "labels" not in lbl_p.parts:
            continue
        for ext in [".jpg", ".jpeg", ".png"]:
            possible_img = lbl_p.parent.parent / "images" / (lbl_p.stem + ext)
            if possible_img.is_file():
                boxes = []
                with open(lbl_p, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            raw_cid = int(parts[0])
                            raw_name = archive_classes[raw_cid] if raw_cid < len(archive_classes) else str(raw_cid)
                            cid, cname, status = map_raw_label(raw_name)
                            if status == "accepted":
                                boxes.append({
                                    "class_id": cid,
                                    "class_name": cname,
                                    "bbox": [float(p) for p in parts[1:]],
                                    "annotation_type": "manual_bbox",
                                    "original_label": raw_name
                                })
                            else:
                                flagged_ambiguous_log.append({"file": lbl_p.name, "raw": raw_name, "status": status})
                if boxes:
                    # Group by sequence stem (e.g. dam_001)
                    seq_group = "dam_" + lbl_p.stem.split("_")[0]
                    candidates.append({
                        "id": lbl_p.stem,
                        "source": "damsegment",
                        "group_id": seq_group,
                        "img_path": possible_img,
                        "annotations": boxes,
                        "annotation_type": "manual_bbox"
                    })
                break

# 3. Ingest Roboflow Heritage if extracted
heritage_dir = EXTRACT_DIR / "roboflow_heritage"
if heritage_dir.is_dir():
    archive_classes = archive_inspections.get("roboflow_heritage", {}).get("classes_found", [])
    for lbl_p in heritage_dir.rglob("*.txt"):
        if "labels" not in lbl_p.parts:
            continue
        for ext in [".jpg", ".jpeg", ".png"]:
            possible_img = lbl_p.parent.parent / "images" / (lbl_p.stem + ext)
            if possible_img.is_file():
                boxes = []
                with open(lbl_p, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            raw_cid = int(parts[0])
                            raw_name = archive_classes[raw_cid] if raw_cid < len(archive_classes) else str(raw_cid)
                            cid, cname, status = map_raw_label(raw_name)
                            if status == "accepted":
                                boxes.append({
                                    "class_id": cid,
                                    "class_name": cname,
                                    "bbox": [float(p) for p in parts[1:]],
                                    "annotation_type": "manual_bbox",
                                    "original_label": raw_name
                                })
                            else:
                                flagged_ambiguous_log.append({"file": lbl_p.name, "raw": raw_name, "status": status})
                if boxes:
                    # Roboflow grouping: base image stem before '.rf.'
                    base_stem = lbl_p.stem.split(".rf.")[0] if ".rf." in lbl_p.stem else lbl_p.stem
                    candidates.append({
                        "id": lbl_p.stem,
                        "source": "roboflow_heritage",
                        "group_id": f"rf_heritage_{base_stem}",
                        "img_path": possible_img,
                        "annotations": boxes,
                        "annotation_type": "manual_bbox"
                    })
                break

# 4. Ingest Roboflow Internal Wall if extracted
internal_wall_dir = EXTRACT_DIR / "roboflow_internal_wall"
if internal_wall_dir.is_dir():
    archive_classes = archive_inspections.get("roboflow_internal_wall", {}).get("classes_found", [])
    for lbl_p in internal_wall_dir.rglob("*.txt"):
        if "labels" not in lbl_p.parts:
            continue
        for ext in [".jpg", ".jpeg", ".png"]:
            possible_img = lbl_p.parent.parent / "images" / (lbl_p.stem + ext)
            if possible_img.is_file():
                boxes = []
                with open(lbl_p, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            raw_cid = int(parts[0])
                            raw_name = archive_classes[raw_cid] if raw_cid < len(archive_classes) else str(raw_cid)
                            cid, cname, status = map_raw_label(raw_name)
                            if status == "accepted":
                                boxes.append({
                                    "class_id": cid,
                                    "class_name": cname,
                                    "bbox": [float(p) for p in parts[1:]],
                                    "annotation_type": "manual_bbox",
                                    "original_label": raw_name
                                })
                            else:
                                flagged_ambiguous_log.append({"file": lbl_p.name, "raw": raw_name, "status": status})
                if boxes:
                    base_stem = lbl_p.stem.split(".rf.")[0] if ".rf." in lbl_p.stem else lbl_p.stem
                    candidates.append({
                        "id": lbl_p.stem,
                        "source": "roboflow_internal_wall",
                        "group_id": f"rf_internal_{base_stem}",
                        "img_path": possible_img,
                        "annotations": boxes,
                        "annotation_type": "manual_bbox"
                    })
                break

print(f"Total raw candidate images loaded across all available sources: {len(candidates)}")
print(f"Flagged/pruned ambiguous annotations: {len(flagged_ambiguous_log)}")

# Perceptual Deduplication
seen_md5 = {}
seen_phash = []
exact_dups = []
near_dups = []
deduplicated_candidates = []

for item in candidates:
    md5, phash = compute_image_hashes(item["img_path"])
    if md5 is None:
        continue
    if md5 in seen_md5:
        exact_dups.append((item["img_path"], seen_md5[md5]))
        continue
    is_near = False
    for prev_path, prev_phash in seen_phash:
        if (phash - prev_phash) <= 2:
            near_dups.append((item["img_path"], prev_path))
            is_near = True
            break
    if not is_near:
        seen_md5[md5] = item["img_path"]
        seen_phash.append((item["img_path"], phash))
        deduplicated_candidates.append(item)

print(f"\nDeduplication Results:")
print(f"  Exact duplicates removed: {len(exact_dups)}")
print(f"  Near duplicates removed:  {len(near_dups)}")
print(f"  Clean unique images:      {len(deduplicated_candidates)}")


## Step 9: Leakage-Resistant Group-Aware Splitting

Splits candidates into `train` (70%), `val` (15%), and `test` (15%) splits, ensuring that all images sharing a `group_id` stay in the same split.  
**Fixed Algorithm:** Correctly balances multi-group and single-group allocations without dumping all images into test.


In [ ]:
# 9. Group-aware splitting
import random
from collections import defaultdict

def group_aware_split(dataset_items, train_ratio=0.70, val_ratio=0.15, seed=42):
    random.seed(seed)
    if not dataset_items:
        return {"train": [], "val": [], "test": []}
        
    groups = defaultdict(list)
    for item in dataset_items:
        groups[item['group_id']].append(item)
        
    group_keys = list(groups.keys())
    random.shuffle(group_keys)
    
    total_items = len(dataset_items)
    target_train = max(1, int(total_items * train_ratio))
    target_val = max(1, int(total_items * val_ratio))
    
    splits = {"train": [], "val": [], "test": []}
    train_count = 0
    val_count = 0
    
    for gk in group_keys:
        items = groups[gk]
        # Prefer train until quota reached
        if train_count < target_train or (train_count + len(items) <= target_train):
            splits["train"].extend(items)
            train_count += len(items)
        elif val_count < target_val or (val_count + len(items) <= target_val):
            splits["val"].extend(items)
            val_count += len(items)
        else:
            splits["test"].extend(items)
            
    # Fallback sanity check: ensure no split is starved if multiple groups exist
    if len(groups) >= 3:
        if not splits["val"] and len(splits["train"]) > 2:
            moved = splits["train"].pop()
            splits["val"].append(moved)
        if not splits["test"] and len(splits["train"]) > 2:
            moved = splits["train"].pop()
            splits["test"].append(moved)
            
    print(f"Group split results ({len(groups)} distinct property groups):")
    for s_name, s_items in splits.items():
        pct = (len(s_items) / total_items * 100) if total_items > 0 else 0
        print(f"  Split '{s_name}': {len(s_items)} images ({pct:.1f}%)")
    return splits

splits = group_aware_split(deduplicated_candidates)


## Step 10: Ultralytics YOLO Directory Assembly & `data.yaml` Generation

Assemble processed images, YOLO label files, `data.yaml`, and a comprehensive provenance manifest (`dataset_manifest.json`).


In [ ]:
# 10. Assembly and YAML generation
def write_yolo_dataset(splits, output_root: Path):
    for split in ["train", "val", "test"]:
        (output_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_root / "labels" / split).mkdir(parents=True, exist_ok=True)
        
    manifest = []
    
    for split_name, items in splits.items():
        for item in items:
            img_src = Path(item["img_path"])
            dest_img_name = f"{item['source']}_{item['id']}{img_src.suffix.lower()}"
            dest_lbl_name = f"{item['source']}_{item['id']}.txt"
            
            dest_img_path = output_root / "images" / split_name / dest_img_name
            dest_lbl_path = output_root / "labels" / split_name / dest_lbl_name
            
            # Copy image
            shutil.copy2(img_src, dest_img_path)
            
            # Write YOLO labels
            with open(dest_lbl_path, "w", encoding="utf-8") as lf:
                for ann in item["annotations"]:
                    cid = ann["class_id"]
                    xc, yc, w, h = ann["bbox"]
                    lf.write(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
                    
            manifest.append({
                "filename": dest_img_name,
                "split": split_name,
                "source": item["source"],
                "group_id": item["group_id"],
                "defect_count": len(item["annotations"]),
                "classes": [a["class_id"] for a in item["annotations"]],
                "annotation_type": item.get("annotation_type", "manual_bbox"),
                "original_label": item.get("original_label", "")
            })
            
    # Generate data.yaml
    yaml_dict = {
        "path": str(output_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": CANONICAL_CLASSES
    }
    
    with open(output_root / "data.yaml", "w", encoding="utf-8") as yf:
        yaml.safe_dump(yaml_dict, yf, sort_keys=False)
        
    with open(output_root / "dataset_manifest.json", "w", encoding="utf-8") as mf:
        json.dump(manifest, mf, indent=2)
        
    print(f"[COMPLETED] Dataset assembled at: {output_root}")
    print(f"Total manifest records: {len(manifest)}")
    return yaml_dict

if sum(len(s) for s in splits.values()) > 0:
    yolo_config = write_yolo_dataset(splits, PROCESSED_DIR)
else:
    print("[NOTICE] Zero candidates available to assemble yet. Check download status in Step 4.")


## Step 11: Ground-Truth Label Visualization Sanity Check

Renders sample images with ground-truth bounding boxes and canonical class labels to visually verify coordinate alignment, color coding, and defect localization.


In [ ]:
# 11. Ground-truth visualization helper
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CLASS_COLORS = {
    0: "#E63946",  # Red: Wall Crack
    1: "#457B9D",  # Blue: Dampness / Water Damage
    2: "#E9C46A",  # Yellow: Peeling Paint / Plaster
    3: "#F4A261",  # Orange: Spalling / Surface Damage
}

def visualize_sample_annotations(image_path: Path, label_path: Path):
    if not image_path.is_file() or not label_path.is_file():
        return
    img = cv2.imread(str(image_path))
    if img is None:
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape
    
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img_rgb)
    ax.set_title(f"Ground-Truth Inspection: {image_path.name}", fontsize=12)
    ax.axis("off")
    
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cid = int(parts[0])
                xc, yc, bw, bh = [float(v) for v in parts[1:]]
                x1 = (xc - bw / 2.0) * w
                y1 = (yc - bh / 2.0) * h
                box_w = bw * w
                box_h = bh * h
                color = CLASS_COLORS.get(cid, "#00FF00")
                cname = CANONICAL_CLASSES.get(cid, f"Class {cid}")
                
                rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, max(y1 - 5, 0), cname, color="white", fontsize=9,
                        bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', pad=2))
    plt.tight_layout()
    plt.show()

# Sample from generated dataset
train_imgs = list((PROCESSED_DIR / "images" / "train").glob("*.*"))
if train_imgs:
    sample_img = train_imgs[0]
    sample_lbl = PROCESSED_DIR / "labels" / "train" / (sample_img.stem + ".txt")
    visualize_sample_annotations(sample_img, sample_lbl)
else:
    print("No processed images available in train split for visualization.")


## Step 12: Final Dataset Validation and Pre-Training Readiness Gate

Performs exhaustive audits on:
- Per-class image and annotation counts for each split.
- Empty label files, missing files, and duplicate hashes.
- Group-aware disjointness check across splits.
- Provenance breakdown (manual bounding boxes vs mask-derived bounding boxes).
- **STRICT GATE:** Asserts non-empty splits and verified annotations across ALL 4 canonical classes. If any check fails, marks dataset **`NOT READY FOR TRAINING`** and blocks training.


In [ ]:
# 12. Final automated validation check and strict readiness gate
def audit_and_validate_readiness(dataset_root: Path):
    yaml_path = dataset_root / "data.yaml"
    if not yaml_path.is_file():
        print(f"[FATAL ERROR] data.yaml not found at {yaml_path}")
        return False, ["Missing data.yaml"]
        
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
        
    manifest_path = dataset_root / "dataset_manifest.json"
    manifest_records = []
    if manifest_path.is_file():
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest_records = json.load(f)
            
    # Metrics containers
    split_image_counts = {}
    split_empty_labels = {}
    split_class_boxes = {s: {cid: 0 for cid in CANONICAL_CLASSES} for s in ["train", "val", "test"]}
    split_class_images = {s: {cid: 0 for cid in CANONICAL_CLASSES} for s in ["train", "val", "test"]}
    provenance_counts = defaultdict(int)
    annotation_type_counts = defaultdict(int)
    split_groups = defaultdict(set)
    validation_errors = []
    
    for split in ["train", "val", "test"]:
        img_dir = dataset_root / "images" / split
        lbl_dir = dataset_root / "labels" / split
        if not img_dir.is_dir() or not lbl_dir.is_dir():
            split_image_counts[split] = 0
            validation_errors.append(f"Directory missing for split '{split}'")
            continue
            
        imgs = sorted(list(img_dir.glob("*.*")))
        lbls = sorted(list(lbl_dir.glob("*.txt")))
        split_image_counts[split] = len(imgs)
        split_empty_labels[split] = 0
        
        # 1-to-1 matching assert
        img_stems = {p.stem for p in imgs}
        lbl_stems = {p.stem for p in lbls}
        missing_lbls = img_stems - lbl_stems
        orphan_lbls = lbl_stems - img_stems
        if missing_lbls:
            validation_errors.append(f"{split}: {len(missing_lbls)} images lack label files.")
        if orphan_lbls:
            validation_errors.append(f"{split}: {len(orphan_lbls)} orphan labels lack images.")
            
        for lf in lbls:
            with open(lf, "r", encoding="utf-8") as f:
                lines = [l.strip() for l in f if l.strip()]
            if not lines:
                split_empty_labels[split] += 1
                continue
            seen_in_file = set()
            for line in lines:
                parts = line.split()
                if len(parts) == 5:
                    cid = int(parts[0])
                    coords = [float(v) for v in parts[1:]]
                    if not (0 <= cid < 4):
                        validation_errors.append(f"Invalid class ID {cid} in {lf.name}")
                    if not all(0.0 <= v <= 1.0 for v in coords):
                        validation_errors.append(f"Out of bounds bbox in {lf.name}: {coords}")
                    if coords[2] <= 0 or coords[3] <= 0:
                        validation_errors.append(f"Zero dimension bbox in {lf.name}")
                    split_class_boxes[split][cid] += 1
                    seen_in_file.add(cid)
            for cid in seen_in_file:
                split_class_images[split][cid] += 1

    # Audit Manifest for groups and provenance
    for rec in manifest_records:
        split_groups[rec["split"]].add(rec["group_id"])
        provenance_counts[rec["source"]] += 1
        annotation_type_counts[rec["annotation_type"]] += 1
        
    # Group Overlap Check
    train_val_overlap = split_groups["train"] & split_groups["val"]
    train_test_overlap = split_groups["train"] & split_groups["test"]
    val_test_overlap = split_groups["val"] & split_groups["test"]
    if train_val_overlap:
        validation_errors.append(f"Group leakage between train and val: {len(train_val_overlap)} shared groups.")
    if train_test_overlap:
        validation_errors.append(f"Group leakage between train and test: {len(train_test_overlap)} shared groups.")
    if val_test_overlap:
        validation_errors.append(f"Group leakage between val and test: {len(val_test_overlap)} shared groups.")
        
    # Class coverage check
    total_class_boxes = {cid: sum(split_class_boxes[s][cid] for s in ["train", "val", "test"]) for cid in CANONICAL_CLASSES}
    empty_classes = [CANONICAL_CLASSES[cid] for cid, cnt in total_class_boxes.items() if cnt == 0]
    if empty_classes:
        validation_errors.append(f"Missing required defect classes (0 annotations): {empty_classes}")
        
    # Split non-empty checks
    if split_image_counts.get("train", 0) == 0:
        validation_errors.append("Split 'train' is completely EMPTY.")
    if split_image_counts.get("val", 0) == 0:
        validation_errors.append("Split 'val' is completely EMPTY.")
    if split_image_counts.get("test", 0) == 0:
        validation_errors.append("Split 'test' is completely EMPTY.")

    # Print Detailed Audit Report
    print("=" * 80)
    print("                    FINAL DATASET AUDIT & PROVENANCE REPORT")
    print("=" * 80)
    print(f"\n1. Splits Summary:")
    for s in ["train", "val", "test"]:
        imgs_n = split_image_counts.get(s, 0)
        empty_n = split_empty_labels.get(s, 0)
        grps_n = len(split_groups.get(s, set()))
        print(f"  Split '{s:<5}': {imgs_n:>5,} images | {empty_n:>4} clean negative images | {grps_n:>4} property groups")
        
    print(f"\n2. Per-Split Class Instance Breakdown (Bounding Boxes):")
    print(f"  {'Class ID & Name':<32} | {'Train':<10} | {'Val':<10} | {'Test':<10} | {'Total':<10}")
    print("  " + "-" * 76)
    for cid in sorted(CANONICAL_CLASSES.keys()):
        cname = f"Class {cid} ({CANONICAL_CLASSES[cid]})"
        t_b = split_class_boxes["train"][cid]
        v_b = split_class_boxes["val"][cid]
        te_b = split_class_boxes["test"][cid]
        tot = total_class_boxes[cid]
        print(f"  {cname:<32} | {t_b:>10,} | {v_b:>10,} | {te_b:>10,} | {tot:>10,}")
    print("  " + "-" * 76)
    
    print(f"\n3. Annotation Provenance Breakdown:")
    for src, cnt in provenance_counts.items():
        print(f"  Source: {src:<25} -> {cnt:,} images")
    print(f"\n4. Annotation Lineage:")
    for atype, cnt in annotation_type_counts.items():
        print(f"  Type:   {atype:<25} -> {cnt:,} records")
        
    print(f"\n5. Leakage & Group Integrity:")
    print(f"  Train/Val Group Overlap:  {len(train_val_overlap)} (Must be 0)")
    print(f"  Train/Test Group Overlap: {len(train_test_overlap)} (Must be 0)")
    print(f"  Val/Test Group Overlap:   {len(val_test_overlap)} (Must be 0)")
    
    # Final Readiness Evaluation
    print("\n" + "=" * 80)
    if validation_errors:
        print("  DATASET READINESS STATUS: [NOT READY FOR TRAINING]")
        print("=" * 80)
        print("  CRITICAL FAILURES PREVENTING TRAINING:")
        for err in validation_errors:
            print(f"    ❌ {err}")
        print("\n  REQUIRED ACTIONS:")
        print("    1. If missing classes: Provide Mendeley DamSegment, Roboflow Heritage, or Roboflow Internal Wall archives.")
        print("    2. If empty train/val splits: Supply multi-group datasets so the splitter can populate all splits.")
        print("    3. Do NOT proceed to training notebook until all 4 classes possess verified instances.")
        print("=" * 80)
        return False, validation_errors
    else:
        print("  DATASET READINESS STATUS: [VERIFIED & READY FOR TRAINING]")
        print("=" * 80)
        print("  All splits non-empty, all 4 canonical classes verified, zero coordinate errors, zero group leakage.")
        print("=" * 80)
        return True, []

is_ready, audit_errors = audit_and_validate_readiness(PROCESSED_DIR)


## Step 13: Dataset Packaging & Google Drive Archiving

Compresses the prepared dataset into `visionguard_home_dataset.zip` and syncs to Google Drive.  
**Gatekeeper:** If Step 12 determined the dataset is `NOT READY FOR TRAINING`, packaging is blocked to prevent accidental deployment of an incomplete dataset.


In [ ]:
# 13. Packaging archive with pre-training readiness gate
def package_and_archive_dataset(dataset_root: Path, archive_dest: Path, dataset_is_ready: bool):
    if not dataset_is_ready:
        print("\n[BLOCKED] Dataset is NOT READY FOR TRAINING. Packaging aborted.")
        print("Resolve missing defect classes and empty splits before creating the production training archive.")
        return False
        
    print(f"Packaging {dataset_root} -> {archive_dest}...")
    with zipfile.ZipFile(archive_dest, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in dataset_root.rglob('*'):
            if file_path.is_file():
                arcname = file_path.relative_to(dataset_root)
                zf.write(file_path, arcname)
    print(f"[ARCHIVED] Dataset ZIP size: {archive_dest.stat().st_size / (1024*1024):.2f} MB")
    
    gdrive_dest = Path("/content/drive/MyDrive/VisionGuard_Artifacts/visionguard_home_dataset.zip")
    if Path("/content/drive/MyDrive").is_dir():
        gdrive_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive_dest, gdrive_dest)
        print(f"[BACKED UP] Dataset successfully copied to Google Drive: {gdrive_dest}")
    else:
        print("[INFO] Google Drive not mounted. Archive saved locally.")
    return True

archive_path = BASE_DIR / "visionguard_home_dataset.zip"
package_and_archive_dataset(PROCESSED_DIR, archive_path, is_ready)
